This notebook shows how to use GVL Predictor to benchmark a model for generating the task progression of the episodes in a dataset.

First, make sure you select the Notebook kernel. Refer to Readme's Quickstart Guide section for creating the virtual environment. In Readme, we did not install the repo as a pip package, because we use `python -m `. Here, it is easier to run as a pip package, so:

In [1]:
!pip install -e ..

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
Obtaining file:///home/yihao/Downloads/software/gvl/wip/gvl-label-maker
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for gvl (pyproject.toml) ... done
  Created wheel for gvl: filename=gvl-0.1.0-0.editable-py3-none-any.whl size=6187 sha256=e334aa37d795b1584dda0fcee36779d445f154e223a8774bf51ca0804d016080
  Stored in directory: /tmp/pip-ephem-wheel-cache-dj_4srdm/wheels/c8/f6/9c/31c599adf0f5364a14aea530ad163f8e20e4c6e0a61a90cfe2
Successfully built gvl
  Attempting uninstall: gvl
    Found existing installation: gvl 0.1.0
    Uninstalling gvl-0.1.0:
      Successfully uninstalled gvl-0.1.0


Basic environment and output setup.

In [2]:
from __future__ import annotations
from pathlib import Path
import os
from dotenv import load_dotenv
from loguru import logger
from tqdm import tqdm

OUTPUT_DIR = Path("results/notebook-benchmark")
PROMPT_LOG_DIR = OUTPUT_DIR / "conversation_history"

load_dotenv(override=True)
logger.info("Environment variables loaded (dotenv)")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["GVL_CONVERSATION_LOG_DIR"] = str(PROMPT_LOG_DIR)

2026-01-21 14:13:08.879 | INFO     | __main__:<module>:12 - Environment variables loaded (dotenv)


Instantiate data loader. The scripts in the `script` folder mostly use Hydra to configure the parameters. There configurations can be seen from `config` folder. However, here, to show what these parameters are, we use constants.

We set the `ANCHORING` constant in the cell. 
- This tells the prediction agent what the important frames look like. 
  - In the original paper, the authors recommended using the first frame. 
  - In OpenGVL, there are options that you may choose first, middle, or last. 
- Here, we extend that to make it support a list. 
- Some of my preliminary experiments, using only the initial frame, showed that the performance may worsen as the task goes towards the end, which was the reason that I believe anchoring both the first and the last can help. However, this has not been valided systematically, other than some visual inspection confirming the second half of the prediction improved.

In [3]:
from dataclasses import dataclass

from gvl.data_loaders.huggingface import HuggingFaceDataLoader
from gvl.mapper.gemini_mapper import GeminiMapper

# Dataset
DATASET_NAME = "lerobot/aloha_static_towel"
CAMERA_INDEX = 0
MAX_EPISODES = 50
NUM_FRAMES = 15
NUM_CONTEXT_EPISODES = 2

# Data loader
SEED = 42
SHUFFLE = True
SAMPLING_METHOD = "random"
ANCHORING = ["first", "middle", "last"]

data_loader = HuggingFaceDataLoader(
    dataset_name=DATASET_NAME,
    camera_index=CAMERA_INDEX,
    num_frames=NUM_FRAMES,
    num_context_episodes=NUM_CONTEXT_EPISODES,
    shuffle=SHUFFLE,
    seed=SEED,
    max_episodes=MAX_EPISODES,
    sampling_method=SAMPLING_METHOD,
    anchoring=ANCHORING,
)


/home/yihao/miniconda3/envs/gvl-test2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


We start with building a mapper. A mapper is an agent that parses some unstructured raw text outputs from the upstream agents. In this case, we need the progression score(s) of the frame(s). A raw output from the prediction agent usually contains sentences that are part of the agent's analysis, which is not needed. You may set the prediction agent's persona to avoid that, but here, we use a specialized agent, the mapper, to do that.

In [4]:
# Mapper
MAPPER_MODEL_NAME = "gemini-2.5-flash-lite"
MAPPER_MAX_NEW_TOKENS = 1024
MAPPER_TEMPERATURE = 0.75
MAPPER_RETRIES = 3
MAPPING_PROMPT_TEMPLATE = """You are a specialized data extractor. Your sole purpose is to identify and extract task completion percentages for given frames from the provided text.

**Instructions:**

1.  **Identify Percentages:** Scan the user's message and identify all occurrences of task completion percentages. These are typically expressed as numbers followed by a percent sign (e.g., "5%", "30%", etc.). Don't take into consideration overall task completion, starting frame completion, or any other non-specific percentages. If frame i appears multiple times, extract only one percentage for that frame (the first one).
2.  **Extract Numerical Values:** From each identified percentage, extract only the numerical value. For example, from "5%", you will extract the number `5`.
3.  **Format as JSON:** Compile all extracted numerical values into a single JSON object. The JSON object must be in the following format: `{"prediction": [list_of_percentages]}`.
4.  **No Percentages Found:** If the user's message contains no percentage values, return an empty list within the JSON object, like this: `{"prediction": []}`.

**Important:** 
- Your response must **only** contain the final JSON object. Do not include any additional text, explanations, apologies, or markdown formatting. 
- Don't take into consideration overall task completion, starting frame completion, or any other non-specific percentages.
- Make sure the number of frames in the list corresponds to the number of unique frames mentioned in the user's message. Do not add extra numbers or omit any frames.

---

### **Example:**

**User Message:**
`As an expert roboticist, I will now analyze each frame to predict the task completion percentage for the task of "open door". The analysis for each frame is independent and based on a decomposition of the task into key stages: approaching the handle, grasping the handle, and pulling the door open.

**Initial State (0% completion):** The robot is positioned before the closed cabinet, ready to begin the task.
**Final State (100% completion):** Frame 1: The robot is approaching the cup. Task Completion: 5% Frame 2: The robot is grasping the cup. Task Completion: 30% Frame 3: The robot is lifting the cup. Task Completion: 60% Frame 4: The robot is pouring the liquid. Task Completion: 80% Frame 5: The liquid is flowing into the cup. Task Completion: 90% Frame 6: The pouring is almost complete. Task Completion: 95% Frame 7: The robot is retracting its arm. Task Completion: 98% Frame 8: The robot is returning to its initial position. Task Completion: 99% Frame 9: The robot is back to its initial position. Task Completion: 99% Frame 10: The robot is back to its initial position. Task Completion: 99% Frame 11: The robot is back to its initial position. Task Completion: 99% Frame 12: The robot is back to its initial position. Task Completion: 99% Frame 13: The robot is back to its initial position. Task Completion: 99% Frame 14: The robot is back to its initial position. Task Completion: 99% Frame 15: The robot is back to its initial position. Task Completion: 99% Frame 16: The robot is back to its initial position. Task Completion: 99% Frame 17: The robot is back to its initial position. Task Completion: 99% Frame 18: The robot is back to its initial position. Task Completion: 99% Frame 19: The robot is back to its initial position. Task Completion: 99% Frame 20: The robot is back to its initial position. Task Completion: 100%. Final Task Completion Percentage: 88.0%`

**Your Response:**
```json
{"prediction": [5, 30, 60, 80, 90, 95, 98, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 99, 100]}
```

Answer:
"""

@dataclass
class MappingPrompt:
    template: str
    name: str = "default"

mapping_prompt = MappingPrompt(template=MAPPING_PROMPT_TEMPLATE)
mapper = GeminiMapper(
    model_name=MAPPER_MODEL_NAME,
    max_new_tokens=MAPPER_MAX_NEW_TOKENS,
    temperature=MAPPER_TEMPERATURE,
    retries=MAPPER_RETRIES,
    mapping_prompt=mapping_prompt,
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Then, we build the prediction agent's client.

In [5]:
# Model
from gvl.clients.qwen3 import Qwen3Client
import datetime

MODEL_CLASS = Qwen3Client
MODEL_NAME = "Qwen/Qwen3-VL-32B-Instruct"
MODEL_MAX_INPUT_LENGTH = 32000
client = MODEL_CLASS(model_name=MODEL_NAME, max_input_length=MODEL_MAX_INPUT_LENGTH)

PROMPT_TEMPLATE = """You are an expert roboticist tasked to predict task completion percentages for frames of a robot for the task of {instruction}.
The task completion percentages are between 0 and 100, where 100 corresponds to full task completion.
The frames may be in random order; reason about each frame independently when estimating completion.
Make sure the number of frames in the list corresponds to the number of unique frames mentioned in the user's message in your final answer.
There are {num_frames} frames. Do not add extra numbers or omit any frames.
"""
PROMPT_NAME = "default"

PROMPT_PHRASES = {
    "anchor_scene_label_start": "Initial robot scene:",
    "anchor_scene_completion_start": "In the initial robot scene, the task completion percentage is 0%.",
    "anchor_scene_label_middle": "Middle robot scene:",
    "anchor_scene_completion_middle": "In the middle robot scene, the task completion percentage is 50%.",
    "anchor_scene_label_last": "Last robot scene:",
    "anchor_scene_completion_last": "In the last robot scene, the task completion percentage is 100%.",
    "context_frame_label_template": "Frame {i}:",
    "context_frame_completion_template": "Task Completion Percentage: {p}%",
    "eval_frame_label_template": "Frame {i}:",
    "eval_task_completion_instruction": [
        "Now, for the task of {instruction}, output the task completion percentage for the following frames that are presented in random order. For each frame, format your response as follow: Frame {{i}}: Task Completion Percentages:{{}}%",
        "Be rigorous and precise; percentage reflects task completion. There are {num_frames} frames. Do not add extra numbers or omit any frames.",
        "Remember: frames are in random order.",
    ],
}

Skipping import of cpp extensions due to incompatible torch version 2.7.1+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
2026-01-21 14:13:13.000 | INFO     | gvl.clients.qwen3:__init__:23 - Loading Qwen3 model Qwen/Qwen3-VL-32B-Instruct ...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 14/14 [00:12<00:00,  1.09it/s]
2026-01-21 14:13:28.973 | INFO     | gvl.clients.qwen3:__init__:30 - <class 'transformers.models.qwen3_vl.processing_qwen3_vl.Qwen3VLProcessor'>


We load evaluation cases. Check the load_episode_eval_cases function for how that's done.

In [6]:
from gvl.utils import inference as infer_utils

NUM_EVAL_CASES = 3

eval_cases = infer_utils.load_episode_eval_cases(data_loader, NUM_EVAL_CASES, DATASET_NAME)
logger.info(
    f"Loaded {len(eval_cases)} (in-context trajectories (0 or more) + eval trajectory) eval cases for prediction"
)
if len(eval_cases) == 0:
    logger.warning("No eval cases loaded; exiting")
    raise SystemExit(0)

2026-01-21 14:13:29.126 | INFO     | gvl.utils.inference:load_episode_eval_cases:131 - Generating 3 eval cases…
2026-01-21 14:13:29.126 | INFO     | gvl.utils.inference:load_episode_eval_cases:134 - Loading eval case 1/3
2026-01-21 14:13:29.127 | INFO     | gvl.data_loaders.huggingface:load_fewshot_input:101 - Loading episode 0 from lerobot/aloha_static_towel
2026-01-21 14:13:29.290 | INFO     | gvl.data_loaders.huggingface:_load_episode_frames:57 - Loading episode [0] frames from 0 to 500 (exclusive)
2026-01-21 14:13:31.739 | INFO     | gvl.data_loaders.huggingface:_load_episode_frames:57 - Loading episode [24] frames from 12000 to 12500 (exclusive)
2026-01-21 14:13:34.036 | INFO     | gvl.data_loaders.huggingface:_load_episode_frames:57 - Loading episode [33] frames from 16500 to 17000 (exclusive)
2026-01-21 14:13:36.193 | INFO     | gvl.utils.inference:load_episode_eval_cases:134 - Loading eval case 2/3
2026-01-21 14:13:36.194 | INFO     | gvl.data_loaders.huggingface:load_fewshot_i

Prepare for the benchmark pipeline.

In [7]:
import json
from datetime import datetime

from tqdm import tqdm

from gvl.metrics.voc import VOCMetric
from gvl.results.prediction import aggregate_metrics
from gvl.utils.frame import order_episode_frames, save_frame_visualizations, save_progress_video

SAVE_RAW = True
SAVE_IMAGES = True
SAVE_VIDEOS = True
TEMPERATURE = 1.0
VIDEO_FPS = 2

starting_time = datetime.now().isoformat().replace(":", "-")
model_name_safe = client.model_name.replace("/", "_")
jsonl_path = OUTPUT_DIR / f"{model_name_safe}_{starting_time}_predictions.jsonl"

logger.info(
    f"Instantiated components | dataset={DATASET_NAME} loader={data_loader.__class__.__name__} "
    f"model={client.__class__.__name__} prompt_template_chars={len(PROMPT_TEMPLATE)}"
)

voc_metric = VOCMetric()
logger.debug(f"Metrics initialized: {voc_metric.name}")

frames_dir = None
if SAVE_IMAGES or SAVE_VIDEOS:
    frames_dir = OUTPUT_DIR / f"{model_name_safe}_{starting_time}_frames"
    frames_dir.mkdir(parents=True, exist_ok=True)
    if SAVE_IMAGES:
        logger.info(f"Saving labeled frames to {frames_dir}")
    if SAVE_VIDEOS:
        logger.info(f"Saving videos to {frames_dir}")

2026-01-21 14:13:49.829 | INFO     | __main__:<module>:20 - Instantiated components | dataset=lerobot/aloha_static_towel loader=HuggingFaceDataLoader model=Qwen3Client prompt_template_chars=547
2026-01-21 14:13:49.829 | DEBUG    | __main__:<module>:26 - Metrics initialized: voc
2026-01-21 14:13:49.830 | INFO     | __main__:<module>:33 - Saving labeled frames to results/notebook-benchmark/Qwen_Qwen3-VL-32B-Instruct_2026-01-21T14-13-49.829627_frames
2026-01-21 14:13:49.830 | INFO     | __main__:<module>:35 - Saving videos to results/notebook-benchmark/Qwen_Qwen3-VL-32B-Instruct_2026-01-21T14-13-49.829627_frames


The benchmark loop.

In [8]:
records: list = []
logger.info(f"Streaming prediction records to {jsonl_path}")
with jsonl_path.open("w", encoding="utf-8") as jsonl_file:
    for idx, eval_case in tqdm(enumerate(eval_cases), total=NUM_EVAL_CASES, desc="Predicting"):
        record = infer_utils.predict_on_episode_eval_case(
            idx,
            NUM_EVAL_CASES,
            eval_case,
            client,
            PROMPT_TEMPLATE,
            SAVE_RAW,
            voc_metric,
            DATASET_NAME,
            temperature=float(TEMPERATURE),
            mapper=mapper,
            prompt_phrases=PROMPT_PHRASES,
        )
        records.append(record)
        jsonl_file.write(json.dumps(record.to_dict(include_images=False), ensure_ascii=False) + "\n")
        jsonl_file.flush()

        if SAVE_IMAGES and frames_dir is not None:
            save_frame_visualizations([record], frames_dir)

        if SAVE_VIDEOS and frames_dir is not None:
            eval_ep = record.example.eval_episode
            eval_case_dir = frames_dir / f"eval_case_{record.index:04d}"
            eval_case_dir.mkdir(parents=True, exist_ok=True)
            video_path = eval_case_dir / f"eval_episode_{eval_ep.episode_index}_gt_original.mp4"
            video_frames, video_values = order_episode_frames(
                eval_ep,
                eval_ep.shuffled_frames_approx_completion_rates,
                order="original",
            )
            logger.info(f"Saving ground-truth video in 'original' order at {video_path}")
            try:
                save_progress_video(
                    video_frames,
                    video_values,
                    video_path,
                    label_prefix="gt",
                    fps=VIDEO_FPS,
                )
            except Exception as exc:
                logger.exception(f"Failed to save ground-truth video at {video_path}: {exc}")
                logger.error("MP4 saving requires imageio + imageio-ffmpeg (and ffmpeg).")
            else:
                logger.info(f"Wrote ground-truth video to {video_path}")

            pred_video_path = eval_case_dir / f"eval_episode_{eval_ep.episode_index}_pred_original.mp4"
            pred_frames, pred_values = order_episode_frames(
                eval_ep,
                eval_ep.shuffled_frames_predicted_completion_rates,
                order="original",
            )
            logger.info(f"Saving prediction video in 'original' order at {pred_video_path}")
            try:
                save_progress_video(
                    pred_frames,
                    pred_values,
                    pred_video_path,
                    label_prefix="pred",
                    fps=VIDEO_FPS,
                )
            except Exception as exc:
                logger.exception(f"Failed to save prediction video at {pred_video_path}: {exc}")
                logger.error("MP4 saving requires imageio + imageio-ffmpeg (and ffmpeg).")
            else:
                logger.info(f"Wrote prediction video to {pred_video_path}")

2026-01-21 14:13:49.846 | INFO     | __main__:<module>:2 - Streaming prediction records to results/notebook-benchmark/Qwen_Qwen3-VL-32B-Instruct_2026-01-21T14-13-49.829627_predictions.jsonl
Predicting:   0%|          | 0/3 [00:00<?, ?it/s]2026-01-21 14:13:49.848 | INFO     | gvl.utils.inference:predict_on_episode_eval_case:156 - Processing eval case 1/3 (episode_index=0) from lerobot/aloha_static_towel
2026-01-21 14:13:49.848 | DEBUG    | gvl.utils.inference:_generate_eval_case_response:95 - Prompt (truncated to 400 chars): You are an expert roboticist tasked to predict task completion percentages for frames of a robot for the task of Pick up a piece of paper towel and place it on the spilled liquid..
The task completion percentages are between 0 and 100, where 100 corresponds to full task completion.
The frames may be in random order; reason about each frame independently when estimating completion.
Make sure the nu...
2026-01-21 14:13:49.848 | DEBUG    | gvl.clients.base:_generate_wi

Log the metrics.

In [9]:
dataset_metrics = aggregate_metrics(records)
logger.success(
    f"Aggregate metrics: total={dataset_metrics.total_examples} valid={dataset_metrics.valid_predictions} "
    f"ratio={(dataset_metrics.length_valid_ratio if dataset_metrics.length_valid_ratio is not None else 0.0):.2f} "
    f"voc_mean={dataset_metrics.metric_means.get('voc', float('nan')):.4f}"
)
summary = dict()
summary["model_name"] = client.model_name
summary["dataset_name"] = DATASET_NAME
summary["num_context_episodes"] = NUM_CONTEXT_EPISODES
summary["prediction_time"] = starting_time
summary["temperature"] = float(TEMPERATURE)
summary["num_eval_cases"] = len(records)
summary["sampling"] = SAMPLING_METHOD
summary["metrics"] = dataset_metrics.to_dict()
summary["prompt_type"] = PROMPT_NAME
summary["anchoring"] = ANCHORING

with (OUTPUT_DIR / f"{model_name_safe}_{starting_time}_summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
logger.info(f"Wrote {len(records)} records to {jsonl_path}")
logger.info(f"Summary: {summary}")

2026-01-21 14:14:38.888 | SUCCESS  | __main__:<module>:2 - Aggregate metrics: total=3 valid=3 ratio=1.00 voc_mean=0.9405
2026-01-21 14:14:38.888 | INFO     | __main__:<module>:21 - Wrote 3 records to results/notebook-benchmark/Qwen_Qwen3-VL-32B-Instruct_2026-01-21T14-13-49.829627_predictions.jsonl
2026-01-21 14:14:38.889 | INFO     | __main__:<module>:22 - Summary: {'model_name': 'Qwen/Qwen3-VL-32B-Instruct', 'dataset_name': 'lerobot/aloha_static_towel', 'num_context_episodes': 2, 'prediction_time': '2026-01-21T14-13-49.829627', 'temperature': 1.0, 'num_eval_cases': 3, 'sampling': 'random', 'metrics': {'total_examples': 3, 'valid_predictions': 3, 'length_valid_ratio': 1.0, 'metric_means': {'voc': 0.9405124063264717}}, 'prompt_type': 'default', 'anchoring': ['first', 'middle', 'last']}


Cleanups.

In [10]:
from gvl.utils.cleanup import cleanup_resources

# Uncomment the following line to cleanup resources after prediction
# cleanup_resources(clients=[client, mapper], records=records)
